# Diagnóstico de Calidad de Datos — ETL Vehículos Eléctricos

Este notebook analiza las inconsistencias entre los dos datasets fuente y explica los valores NaN/vacíos en las tablas golden. Corre sobre las capas **bronze** y **silver** del catálogo `catalog_smartdata_final`.

| Dataset | Fuente | Alcance |
|---|---|---|
| `ev_population_data` | WA DOL (registro vehicular del estado de WA) | 210 165 registros, ~43 marcas |
| `ev_specs_2025` | ev-database.org | 478 variantes, ~59 marcas, **mercado global / europeo 2025** |

In [ ]:
dbutils.widgets.removeAll()
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [ ]:
dbutils.widgets.text("catalogo", "catalog_smartdata_final")
catalogo = dbutils.widgets.get("catalogo")

tbl_pop_bronze   = f"{catalogo}.bronze.ev_population_bronze"
tbl_spec_bronze  = f"{catalogo}.bronze.ev_specs_2025_bronze"
tbl_enriched_s   = f"{catalogo}.silver.ev_vehicle_enriched"
tbl_ratio_g      = f"{catalogo}.golden.golden_make_range_price_ratio"
tbl_price_g      = f"{catalogo}.golden.golden_make_price_range"

df_pop   = spark.table(tbl_pop_bronze)
df_specs = spark.table(tbl_spec_bronze)
print(f"Population bronze: {df_pop.count():,} filas")
print(f"Specs bronze     : {df_specs.count():,} filas")

## 1. Calidad del campo `base_msrp`

**Hipótesis**: el MSRP es 0 en casi todos los registros → `avg_price` en golden no es fiable como precio real.

In [ ]:
total_pop = df_pop.count()
msrp_zero = df_pop.filter(F.col("base_msrp") == 0).count()
msrp_pos  = df_pop.filter(F.col("base_msrp") > 0).count()

print(f"Total registros       : {total_pop:>8,}  (100.0%)")
print(f"MSRP = 0 (no reportado): {msrp_zero:>8,}  ({msrp_zero/total_pop*100:.1f}%)")
print(f"MSRP > 0 (tiene precio): {msrp_pos:>8,}  ({msrp_pos/total_pop*100:.1f}%)")
print()
print("Conclusión: base_msrp = 0 es el valor por defecto del DOL de WA.")
print("No significa precio cero — significa 'no informado'.")
print("→ Filtrar siempre con base_msrp > 0 antes de calcular precios promedio.")

# Distribución de MSRP cuando existe
display(
    df_pop.filter(F.col("base_msrp") > 0)
    .groupBy("make")
    .agg(
        F.count(F.lit(1)).alias("registros_con_msrp"),
        F.round(F.avg("base_msrp"), 0).alias("avg_msrp_real")
    )
    .orderBy(F.desc("registros_con_msrp"))
)

## 2. Calidad del campo `electric_range` — unidades y cobertura

**Problema**: `electric_range` es el alcance EPA en **millas**, mientras que `range_km` del archivo de specs está en **kilómetros**. Compararlos directamente es incorrecto.  
**Además**: 56.5 % de registros tiene `electric_range = 0` (campo no reportado o PHEV con modo 0).

In [ ]:
total = df_pop.count()
range_zero = df_pop.filter(F.col("electric_range") == 0).count()
range_pos  = df_pop.filter(F.col("electric_range") > 0).count()

print(f"electric_range = 0 : {range_zero:>8,}  ({range_zero/total*100:.1f}%)")
print(f"electric_range > 0 : {range_pos:>8,}  ({range_pos/total*100:.1f}%)")
print()
print("Distribución de electric_range cuando > 0 (¡en MILLAS EPA!):")

display(
    df_pop.filter(F.col("electric_range") > 0)
    .groupBy("make")
    .agg(
        F.count(F.lit(1)).alias("registros"),
        F.round(F.avg("electric_range"), 1).alias("avg_range_millas"),
        F.round(F.avg("electric_range") * 1.60934, 1).alias("avg_range_km_convertido")
    )
    .orderBy(F.desc("avg_range_millas"))
    .limit(15)
)

## 3. Cobertura del join make+model entre datasets

**Hipótesis**: el join exacto falla porque los nombres de modelo son incompatibles.  
- Population: `'TESLA'` + `'MODEL 3'`  
- Specs: `'TESLA'` + `'MODEL 3 LONG RANGE AWD (HIGHLAND)'`  

Resultado: **0.34 % de match exacto** → `avg_specs_range_km` = NaN en prácticamente toda la tabla golden.

In [ ]:
normalize = lambda c: F.upper(F.trim(F.regexp_replace(F.col(c), r"\s+", " ")))

pop_keys = df_pop.select(
    normalize("make").alias("make"),
    normalize("model").alias("model")
).distinct()

spec_keys = df_specs.select(
    normalize("brand").alias("make"),
    normalize("model").alias("model")
).distinct()

# Match exacto por marca
pop_makes  = df_pop.select(normalize("make").alias("make")).distinct()
spec_makes = df_specs.select(normalize("brand").alias("make")).distinct()

common_makes   = pop_makes.intersect(spec_makes)
only_pop_makes = pop_makes.subtract(spec_makes)

print(f"Marcas en population : {pop_makes.count()}")
print(f"Marcas en specs      : {spec_makes.count()}")
print(f"Marcas que coinciden : {common_makes.count()}")
print(f"Sólo en population   : {only_pop_makes.count()} — NUNCA tendrán specs_range_km")
print()
print("Marcas sólo en population (sin specs disponibles):")
display(only_pop_makes.orderBy("make"))

In [ ]:
# Para las marcas que coinciden, ¿cuántos modelos hacen match exacto?
joined_models = (
    pop_keys.alias("p")
    .join(spec_keys.alias("s"),
          (F.col("p.make") == F.col("s.make")) &
          (F.col("p.model") == F.col("s.model")),
          how="inner")
)

# Match por prefijo (nuevo método)
prefix_models = (
    pop_keys.alias("p")
    .join(spec_keys.alias("s"),
          (F.col("p.make") == F.col("s.make")) &
          F.col("s.model").startswith(F.col("p.model")),
          how="inner")
    .select(F.col("p.make"), F.col("p.model"))
    .distinct()
)

total_combos = pop_keys.count()
exact_combos = joined_models.count()
prefix_combos = prefix_models.count()

print(f"Combinaciones make+model únicas en population : {total_combos}")
print(f"Con match EXACTO                              : {exact_combos}  ({exact_combos/total_combos*100:.1f}%)")
print(f"Con match por PREFIJO (método nuevo)          : {prefix_combos}  ({prefix_combos/total_combos*100:.1f}%)")
print()
print("Modelos con match exacto:")
display(joined_models.orderBy("p.make", "p.model"))

In [ ]:
# Ver las discrepancias de nombre para las marcas más importantes
for marca in ["TESLA", "BMW", "NISSAN", "VOLKSWAGEN", "TOYOTA"]:
    pop_m = [
        r.model for r in pop_keys.filter(F.col("make") == marca).select("model").collect()
    ]
    spec_m = [
        r.model for r in spec_keys.filter(F.col("make") == marca).select("model").collect()
    ]
    print(f"\n{'='*55}")
    print(f"  {marca}")
    print(f"  Population ({len(pop_m)} modelos): {sorted(pop_m)}")
    print(f"  Specs 2025 ({len(spec_m)} modelos): {sorted(spec_m)[:4]} ...")

## 4. Por qué `avg_specs_range_km` = NaN en toda la tabla golden

Cadena de dependencias que produce el NaN:

```
ev_vehicle_enriched.specs_range_km = NULL (join no dio match)
  └→ golden_make_range_price_ratio filtra base_msrp > 0
       └→ AVG(specs_range_km) = NULL porque TODOS son NULL
            └→ range_per_price_ratio = NULL / precio = NaN
```

**Con el join mejorado** (prefijo), la cobertura sube significativamente para marcas como TESLA, BMW, KIA, TOYOTA, VOLKSWAGEN.

In [ ]:
df_enriched = spark.table(tbl_enriched_s)

total_e = df_enriched.count()
has_match   = df_enriched.filter(F.col("has_specs_match")).count()
has_range   = df_enriched.filter(F.col("specs_range_km").isNotNull()).count()
has_msrp    = df_enriched.filter(F.col("has_valid_msrp")).count()
ratio_ready = df_enriched.filter(
    F.col("specs_range_km").isNotNull() & F.col("has_valid_msrp")
).count()

print(f"Total enriched                                   : {total_e:>8,}")
print(f"Con specs match (has_specs_match=True)            : {has_match:>8,}  ({has_match/total_e*100:.2f}%)")
print(f"Con specs_range_km no nulo                        : {has_range:>8,}  ({has_range/total_e*100:.2f}%)")
print(f"Con base_msrp > 0 (has_valid_msrp)                : {has_msrp:>8,}  ({has_msrp/total_e*100:.2f}%)")
print(f"Con AMBOS (para golden_make_range_price_ratio)     : {ratio_ready:>8,}  ({ratio_ready/total_e*100:.2f}%)")

print()
print("Cobertura por marca (después del join mejorado):")
display(
    df_enriched
    .groupBy("make")
    .agg(
        F.count(F.lit(1)).alias("total"),
        F.sum(F.col("has_specs_match").cast("int")).alias("con_specs"),
        F.round(F.sum(F.col("has_specs_match").cast("int")) / F.count(F.lit(1)) * 100, 1).alias("pct_cobertura")
    )
    .orderBy(F.desc("total"))
)

## 5. Sesgo geográfico: el dataset es casi exclusivamente WA

Este dataset proviene del **DOL (Dept. of Licensing) del estado de Washington**. No es una muestra nacional de EEUU — es el registro vehicular de un solo estado.

In [ ]:
normalize = lambda c: F.upper(F.trim(F.regexp_replace(F.col(c), r"\s+", " ")))
total = df_pop.count()
display(
    df_pop
    .groupBy(normalize("state").alias("state"))
    .agg(
        F.count(F.lit(1)).alias("registros"),
        F.round(F.count(F.lit(1)) / total * 100, 2).alias("pct")
    )
    .orderBy(F.desc("registros"))
)

## 6. Resumen de hallazgos y acciones correctivas

| # | Hallazgo | Impacto | Corrección aplicada |
|---|---|---|---|
| 1 | Join exacto `make+model` → 0.34 % de match | `avg_specs_range_km` = NaN en toda la golden | **Join por prefijo** en `transform_vehicles.ipynb`: si el modelo de population es prefijo del modelo en specs, se hace match y se promedian los specs |
| 2 | 16 marcas sin specs (CHEVROLET, RIVIAN, NISSAN LEAF…) | Sin cobertura estructural | Aceptado: la DB de specs cubre modelos globales 2025; modelos US legacy not included |
| 3 | `base_msrp = 0` en 98.4 % de registros | `avg_price` no refleja precio real | Añadir columna `has_valid_msrp`; filtrar con `base_msrp > 0` en análisis de precios |
| 4 | `electric_range` en **millas** (EPA), `range_km` en **km** | Comparación directa incorrecta | Añadir columna `electric_range_km = electric_range * 1.60934` en silver |
| 5 | 99.8 % de registros son de WA | `golden_state_distribution` no representa EEUU | Aceptado: contexto del dataset (WA DOL) |
| 6 | Specs de 2025 no incluye modelos clásicos (LEAF, BOLT…) | Esas versiones nunca harán match | Documentar; considerar agregar `ev-database.org` histórico si se necesita cobertura completa |